# Lesson 12 — SPI 事务必须由片选划定边界

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


SPI 没有地址；片选（CS）决定哪些字节属于哪个设备。若在 CS 未选中时写显示内容，设备不该悄悄接受。

## 最小知识

本课文本事务是 `CS:LOW\n`、`WRITE:文本\n`、`CS:HIGH\n`。只有在 LOW 到 HIGH 的完整事务结束时，虚拟显示器更新 `display`。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 12-spi-display guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("WRITE:", 0) == 0)
                b_.output.display = b_.input.text.substr(6);
        }

    private:
        Device& b_;
    };
}


## 2. 修改一次

允许一个事务有多次 WRITE，并确认显示内容如何拼接。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 12-spi-display modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("WRITE:", 0) == 0)
                b_.output.display = b_.input.text.substr(6);
        }

    private:
        Device& b_;
    };
}


## 3. 故障实验（Debug）

下面代码收到 WRITE 就立刻改显示器，完全忽略 CS。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 12-spi-display broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("WRITE:", 0) == 0)
                b_.output.display = b_.input.text.substr(6);
        }

    private:
        Device& b_;
    };
}


## 4. 你来完成（Build）

维护 selected 和 frame 状态。未选中 WRITE 或 fault 要设置 error；拉高 CS 才提交 frame。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/12-spi-display/hints.md)。


In [ ]:
%%academy_lab 12-spi-display exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: CS:LOW / WRITE / CS:HIGH 事务
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

设计 command/data 位与一个简单显示器初始化序列。为什么真实 SPI Driver 通常一次提交一整个 transaction？

## 你刚刚真正学到了什么？

SPI 的帧边界由 CS 定义。把选择、字节累积和提交分开，Driver 才能表达真正的总线事务。

## 硬件迁移

STEP 2 需要核对 MOSI/MISO/SCLK/CS 接线、模式、频率和显示器初始化序列；此处仅模拟事务语义。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/12-spi-display-solution.ipynb)。
